# 🎯 AI Recruiter Copilot
### LangChain · Groq Llama 3 70B · FAISS · SQLite · Gradio
**Self-contained Colab notebook — no file uploads needed.**

---
**Run cells top-to-bottom on first launch.**

## ⬇️ Step 1 – Install Dependencies

In [ ]:
!pip install -q langchain langchain-groq langchain-community langchain-huggingface \
    faiss-cpu sentence-transformers gradio faker python-dotenv sqlalchemy \
    pandas numpy reportlab pypdf plotly openai tiktoken
print('✅ Dependencies installed')

## 🔑 Step 2 – Set Your Groq API Key
Get a **free** key at https://console.groq.com

In [ ]:
import os

os.environ['GROQ_API_KEY']     = 'gsk_YOUR_KEY_HERE'   # ← paste your key
os.environ['GROQ_MODEL']       = 'llama-3.3-70b-versatile'   # updated model
os.environ['DB_PATH']          = 'data/recruiter.db'
os.environ['FAISS_INDEX_PATH'] = 'faiss_index'
os.environ['EXPORT_PATH']      = 'exports'
os.environ['EMBEDDING_MODEL']  = 'sentence-transformers/all-MiniLM-L6-v2'

os.makedirs('data', exist_ok=True)
os.makedirs('faiss_index', exist_ok=True)
os.makedirs('exports', exist_ok=True)

print('✅ Environment configured')
print(f'   Model: {os.environ["GROQ_MODEL"]}')

## 🗄️ Step 3 – Database Layer (SQLite)

In [ ]:
import sqlite3, json
from datetime import datetime

DB_PATH = os.environ['DB_PATH']

def get_conn():
    os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

def init_db():
    conn = get_conn()
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS candidates (
        candidate_id TEXT PRIMARY KEY, name TEXT, email TEXT, phone TEXT,
        location TEXT, experience_years REAL, current_ctc REAL, expected_ctc REAL,
        skills TEXT, about_section TEXT, projects TEXT, work_experience TEXT,
        industry TEXT, education TEXT, role TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS engagement_data (
        candidate_id TEXT PRIMARY KEY, response_rate REAL, reply_speed_hours REAL,
        interview_attendance REAL, application_completion REAL, engagement_score REAL
    );
    CREATE TABLE IF NOT EXISTS jobs (
        job_id TEXT PRIMARY KEY, jd_text TEXT, structured_jd TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS job_candidates (
        id INTEGER PRIMARY KEY AUTOINCREMENT, job_id TEXT, candidate_id TEXT,
        status TEXT DEFAULT 'Recommended', match_score REAL DEFAULT 0,
        confidence_score REAL DEFAULT 0, engagement_score REAL DEFAULT 0,
        final_score REAL DEFAULT 0, assessment_score REAL DEFAULT 0,
        updated_score REAL DEFAULT 0,
        updated_at TEXT DEFAULT CURRENT_TIMESTAMP,
        UNIQUE(job_id, candidate_id)
    );
    CREATE TABLE IF NOT EXISTS recruiter_memory (
        id INTEGER PRIMARY KEY AUTOINCREMENT, job_id TEXT, candidate_id TEXT,
        question TEXT, answer TEXT, recruiter_notes TEXT,
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS workflow_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT, job_id TEXT, candidate_id TEXT,
        status TEXT, notes TEXT, created_at TEXT DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS feedback (
        id INTEGER PRIMARY KEY AUTOINCREMENT, job_id TEXT, candidate_id TEXT,
        outcome TEXT, notes TEXT, created_at TEXT DEFAULT CURRENT_TIMESTAMP
    );
    CREATE TABLE IF NOT EXISTS job_assessments (
        id INTEGER PRIMARY KEY AUTOINCREMENT, job_id TEXT, candidate_id TEXT,
        question TEXT, answer TEXT, assessment_score REAL DEFAULT 0,
        score_impact REAL DEFAULT 0, verdict TEXT DEFAULT '',
        feedback TEXT DEFAULT '', targets_requirement TEXT DEFAULT '',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP
    );
    """)
    conn.commit(); conn.close()
    print('✅ Database initialized')

def _parse(row):
    d = dict(row)
    for k in ('skills','projects','work_experience'):
        if isinstance(d.get(k), str):
            try: d[k] = json.loads(d[k])
            except: d[k] = []
    return d

def insert_candidate(c):
    conn = get_conn()
    conn.execute("""
        INSERT OR REPLACE INTO candidates
        (candidate_id,name,email,phone,location,experience_years,current_ctc,
         expected_ctc,skills,about_section,projects,work_experience,industry,education,role)
        VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
    """, (c['candidate_id'],c['name'],c.get('email',''),c.get('phone',''),
          c['location'],c['experience_years'],c['current_ctc'],c['expected_ctc'],
          json.dumps(c['skills']),c['about_section'],json.dumps(c['projects']),
          json.dumps(c['work_experience']),c['industry'],c.get('education',''),
          c.get('role','')))
    conn.commit(); conn.close()

def insert_engagement(e):
    conn = get_conn()
    conn.execute("""
        INSERT OR REPLACE INTO engagement_data
        (candidate_id,response_rate,reply_speed_hours,interview_attendance,
         application_completion,engagement_score) VALUES(?,?,?,?,?,?)
    """, (e['candidate_id'],e['response_rate'],e['reply_speed_hours'],
          e['interview_attendance'],e['application_completion'],e['engagement_score']))
    conn.commit(); conn.close()

def get_candidate(cid):
    conn = get_conn()
    row = conn.execute('SELECT * FROM candidates WHERE candidate_id=?',(cid,)).fetchone()
    conn.close()
    return _parse(row) if row else None

def get_all_candidates():
    conn = get_conn()
    rows = conn.execute('SELECT * FROM candidates').fetchall()
    conn.close()
    return [_parse(r) for r in rows]

def get_candidates_by_ids(ids):
    if not ids: return []
    ph = ','.join('?'*len(ids))
    conn = get_conn()
    rows = conn.execute(f'SELECT * FROM candidates WHERE candidate_id IN ({ph})', ids).fetchall()
    conn.close()
    return [_parse(r) for r in rows]

def get_engagement(cid):
    conn = get_conn()
    row = conn.execute('SELECT * FROM engagement_data WHERE candidate_id=?',(cid,)).fetchone()
    conn.close()
    return dict(row) if row else None

def save_job(job_id, jd_text, structured_jd):
    conn = get_conn()
    conn.execute('INSERT OR REPLACE INTO jobs(job_id,jd_text,structured_jd) VALUES(?,?,?)',
                 (job_id, jd_text, json.dumps(structured_jd)))
    conn.commit(); conn.close()

def get_job(job_id):
    conn = get_conn()
    row = conn.execute('SELECT * FROM jobs WHERE job_id=?',(job_id,)).fetchone()
    conn.close()
    if not row: return None
    d = dict(row)
    d['structured_jd'] = json.loads(d['structured_jd']) if d['structured_jd'] else {}
    return d

def upsert_job_candidate(job_id, candidate_id, scores):
    conn = get_conn()
    conn.execute("""
        INSERT INTO job_candidates(job_id,candidate_id,match_score,confidence_score,
                                   engagement_score,final_score,updated_score)
        VALUES(?,?,?,?,?,?,?)
        ON CONFLICT(job_id,candidate_id) DO UPDATE SET
          match_score=excluded.match_score, confidence_score=excluded.confidence_score,
          engagement_score=excluded.engagement_score, final_score=excluded.final_score,
          updated_score=excluded.final_score, updated_at=CURRENT_TIMESTAMP
    """, (job_id,candidate_id,scores.get('match_score',0),scores.get('confidence_score',0),
          scores.get('engagement_score',0),scores.get('final_score',0),scores.get('final_score',0)))
    conn.commit(); conn.close()

def get_job_candidates(job_id):
    conn = get_conn()
    rows = conn.execute("""
        SELECT jc.*, c.name, c.location, c.experience_years, c.skills, c.industry, c.about_section
        FROM job_candidates jc JOIN candidates c ON jc.candidate_id=c.candidate_id
        WHERE jc.job_id=? ORDER BY jc.updated_score DESC
    """, (job_id,)).fetchall()
    conn.close()
    result = []
    for r in rows:
        d = dict(r)
        if isinstance(d.get('skills'), str): d['skills'] = json.loads(d['skills'])
        result.append(d)
    return result

def update_candidate_status(job_id, candidate_id, status, notes=''):
    conn = get_conn()
    conn.execute('UPDATE job_candidates SET status=?,updated_at=CURRENT_TIMESTAMP WHERE job_id=? AND candidate_id=?',
                 (status,job_id,candidate_id))
    conn.execute('INSERT INTO workflow_history(job_id,candidate_id,status,notes) VALUES(?,?,?,?)',
                 (job_id,candidate_id,status,notes))
    conn.commit(); conn.close()

def update_assessment_score(job_id, candidate_id, new_score):
    conn = get_conn()
    conn.execute('UPDATE job_candidates SET assessment_score=?,updated_score=?,updated_at=CURRENT_TIMESTAMP WHERE job_id=? AND candidate_id=?',
                 (new_score,new_score,job_id,candidate_id))
    conn.commit(); conn.close()

def save_recruiter_memory(job_id, candidate_id, question, answer, notes=''):
    conn = get_conn()
    conn.execute('INSERT INTO recruiter_memory(job_id,candidate_id,question,answer,recruiter_notes) VALUES(?,?,?,?,?)',
                 (job_id,candidate_id,question,answer,notes))
    conn.commit(); conn.close()

def get_recruiter_memory(job_id, candidate_id):
    conn = get_conn()
    rows = conn.execute('SELECT * FROM recruiter_memory WHERE job_id=? AND candidate_id=? ORDER BY created_at DESC',
                        (job_id,candidate_id)).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def save_feedback(job_id, candidate_id, outcome, notes=''):
    conn = get_conn()
    conn.execute('INSERT INTO feedback(job_id,candidate_id,outcome,notes) VALUES(?,?,?,?)',
                 (job_id,candidate_id,outcome,notes))
    conn.commit(); conn.close()

def get_feedback_analytics():
    conn = get_conn()
    rows = conn.execute('SELECT outcome,COUNT(*) as count FROM feedback GROUP BY outcome').fetchall()
    conn.close()
    return [dict(r) for r in rows]

def save_assessment(job_id, candidate_id, question, answer, assessment_score, score_impact,
                    verdict='', feedback='', targets_requirement=''):
    conn = get_conn()
    conn.execute(
        """INSERT INTO job_assessments
           (job_id,candidate_id,question,answer,assessment_score,score_impact,
            verdict,feedback,targets_requirement)
           VALUES(?,?,?,?,?,?,?,?,?)""",
        (job_id,candidate_id,question,answer,assessment_score,score_impact,
         verdict,feedback,targets_requirement))
    conn.commit(); conn.close()

def get_assessments_db(job_id, cid):
    conn = get_conn()
    rows = conn.execute(
        'SELECT * FROM job_assessments WHERE job_id=? AND candidate_id=? ORDER BY created_at',
        (job_id,cid)).fetchall()
    conn.close()
    return [dict(r) for r in rows]

print('✅ Database layer ready')

## 👥 Step 4 – Data Generator (500 Candidates)

In [ ]:
import random, uuid
from faker import Faker

fake = Faker('en_IN')
random.seed(42)

ROLES = {
    'Product Analyst':     {'skills':['SQL','Python','Tableau','Power BI','Product Analytics','A/B Testing','Excel','Stakeholder Management','Dashboarding','Mixpanel','Amplitude','Google Analytics'],'industry':['SaaS','E-Commerce','FinTech','EdTech','HealthTech']},
    'Data Scientist':      {'skills':['Python','Machine Learning','NLP','TensorFlow','PyTorch','Scikit-learn','SQL','Statistics','Feature Engineering','PySpark','Pandas'],'industry':['FinTech','HealthTech','SaaS','E-Commerce','AdTech']},
    'Software Engineer':   {'skills':['Python','Java','Go','Microservices','REST APIs','AWS','Docker','Kubernetes','PostgreSQL','Redis','Kafka','System Design','CI/CD'],'industry':['SaaS','FinTech','E-Commerce','Gaming','Cybersecurity']},
    'Frontend Developer':  {'skills':['React','TypeScript','JavaScript','Next.js','CSS','HTML5','Redux','GraphQL','Jest','Figma','Tailwind CSS','Vue.js'],'industry':['SaaS','E-Commerce','EdTech','Media','Gaming']},
    'Backend Developer':   {'skills':['Node.js','Python','Java','Go','PostgreSQL','MongoDB','Redis','Kafka','Docker','Kubernetes','AWS','REST APIs','GraphQL'],'industry':['SaaS','FinTech','E-Commerce','HealthTech','LogisTech']},
    'ML Engineer':         {'skills':['Python','MLOps','TensorFlow','PyTorch','Kubeflow','Docker','Kubernetes','PySpark','SQL','AWS SageMaker','Airflow'],'industry':['FinTech','HealthTech','AdTech','SaaS','Autonomous Systems']},
    'Data Engineer':       {'skills':['Python','Spark','Airflow','dbt','Kafka','Snowflake','BigQuery','SQL','ETL','Delta Lake','AWS','PostgreSQL','Databricks'],'industry':['FinTech','E-Commerce','SaaS','Media','Healthcare']},
    'Product Manager':     {'skills':['Product Strategy','Roadmap Planning','Stakeholder Management','A/B Testing','User Research','SQL','Jira','Agile','Go-to-Market','OKRs'],'industry':['SaaS','E-Commerce','FinTech','EdTech','HealthTech']},
    'DevOps Engineer':     {'skills':['Kubernetes','Docker','Terraform','AWS','GCP','CI/CD','Jenkins','Ansible','Prometheus','Grafana','Linux','GitHub Actions','Helm'],'industry':['SaaS','FinTech','E-Commerce','Cybersecurity','Media']},
    'Business Analyst':    {'skills':['Business Analysis','SQL','Excel','Power BI','Tableau','Requirements Gathering','Stakeholder Management','BPMN','Agile','User Stories'],'industry':['FinTech','Insurance','Healthcare','Retail','Consulting']},
    'Full Stack Developer':{'skills':['React','Node.js','Python','PostgreSQL','MongoDB','REST APIs','Docker','AWS','TypeScript','Redis','GraphQL','Next.js'],'industry':['SaaS','E-Commerce','EdTech','FinTech','Startup']},
}

LOCATIONS  = ['Bangalore','Mumbai','Delhi','Hyderabad','Pune','Chennai','Kolkata','Noida','Gurgaon','Ahmedabad']
EDUCATION  = ['B.Tech CSE – IIT Bombay','B.Tech IT – NIT Trichy','M.Tech Data Science – IIT Delhi',
               'BCA – Pune University','MCA – Delhi University','B.E. Electronics – VTU',
               'MBA – IIM Bangalore','B.Sc CS – Christ University','M.Sc Data Science – BITS Pilani']
COMPANIES  = ['Flipkart','Zomato','Swiggy','Razorpay','CRED','Meesho','Ola','Byju\'s',
               'Freshworks','Zoho','Infosys','Wipro','TCS','PhonePe','Paytm','Nykaa','Dream11']

PROJECT_TEMPLATES = {
    'Product Analyst': [
        ('Customer Churn Prediction Dashboard','Built end-to-end churn prediction pipeline using SQL and Python, visualized through Tableau. Reduced churn by 12%.'),
        ('Funnel Optimization Analytics','Analyzed user funnel drop-offs using Mixpanel and SQL. Implemented A/B tests improving conversion by 18%.'),
        ('Personalized Product Suggestions Engine','Designed recommendation system that increased avg order value by 22% using collaborative filtering and behavioral analytics.'),
        ('Revenue Attribution Model','Built multi-touch attribution model in Python, saving $200K in annual ad spend.'),
    ],
    'Data Scientist': [
        ('Fraud Detection System','Developed real-time fraud detection using XGBoost. Achieved 96% precision at sub-50ms latency.'),
        ('NLP-Based Customer Support Triage','Built BERT-based classifier to auto-route support tickets. Reduced manual triage effort by 70%.'),
        ('Demand Forecasting Engine','Implemented LSTM-based demand forecasting for 10,000+ SKUs improving accuracy by 34%.'),
    ],
    'Software Engineer': [
        ('Distributed Payments Microservice','Architected high-throughput payment service handling 50K TPS using Go, Kafka, and Redis.'),
        ('API Gateway with Rate Limiting','Built API gateway with token-bucket rate limiting. Reduced downstream failures by 85%.'),
        ('Real-time Order Tracking System','Designed WebSocket-based tracking serving 1M+ users at 99.99% uptime.'),
    ],
}

def gen_project(role, skills):
    templates = PROJECT_TEMPLATES.get(role, [])
    if templates:
        title, desc = random.choice(templates)
    else:
        title = f'{role} Platform Redesign'
        desc  = f'Led redesign using {random.choice(skills)}. Improved performance by {random.randint(20,60)}%.'
    return {'project_id':str(uuid.uuid4())[:8],'title':title,'description':desc,
            'skills_used':random.sample(skills,min(3,len(skills))),
            'duration_months':random.randint(3,12),'impact':f'{random.randint(10,60)}% improvement'}

def gen_work_exp(role, exp, industry):
    titles = {'Product Analyst':['Product Analyst','Senior Product Analyst','Lead Analyst'],
              'Data Scientist':['Data Scientist','Senior Data Scientist'],
              'Software Engineer':['Software Engineer','Senior SWE','Staff Engineer']}
    title_list = titles.get(role, [role, f'Senior {role}'])
    entries, remaining = [], max(1, int(exp))
    while remaining > 0:
        tenure = min(remaining, random.randint(1,3))
        remaining -= tenure
        entries.append({'company':random.choice(COMPANIES),'title':random.choice(title_list),
                        'tenure_years':tenure,'industry':industry,
                        'responsibilities':['Led cross-functional analytics initiatives',
                                            'Collaborated with stakeholders on product requirements',
                                            f'Managed delivery of {random.randint(2,8)} projects/year']})
    return entries

def generate_candidates(n=500):
    role_list = list(ROLES.keys())
    candidates = []
    for i in range(n):
        role     = random.choice(role_list)
        rd       = ROLES[role]
        exp      = round(random.uniform(0.5,15),1)
        location = random.choice(LOCATIONS)
        industry = random.choice(rd['industry'])
        skills   = random.sample(rd['skills'], random.randint(4, min(9,len(rd['skills']))))
        name     = fake.name()
        base_ctc = max(3, exp * random.uniform(1.5,3.5))
        cur_ctc  = round(base_ctc + random.uniform(-1,2), 2)
        exp_ctc  = round(cur_ctc * random.uniform(1.1,1.4), 2)
        about    = (f"{name} is a {role} with {int(exp)} years of experience in {industry}. "
                    f"Proficient in {', '.join(skills[:4])}, with a track record of delivering "
                    f"high-impact solutions and strong cross-functional collaboration.")
        candidates.append({
            'candidate_id': f'C{str(i+1).zfill(4)}',
            'name': name, 'email': fake.email(), 'phone': fake.phone_number(),
            'location': location, 'experience_years': exp,
            'current_ctc': cur_ctc, 'expected_ctc': exp_ctc,
            'skills': skills, 'about_section': about,
            'projects': [gen_project(role,skills) for _ in range(random.randint(2,3))],
            'work_experience': gen_work_exp(role, exp, industry),
            'industry': industry, 'education': random.choice(EDUCATION), 'role': role,
        })
    return candidates

def gen_engagement(cid):
    rr  = round(random.uniform(0.2,1.0),2)
    rs  = round(random.uniform(1,72),1)
    ia  = round(random.uniform(0.5,1.0),2)
    ac  = round(random.uniform(0.6,1.0),2)
    score = round((0.4*rr + 0.2*(1-rs/72) + 0.25*ia + 0.15*ac)*100, 2)
    return {'candidate_id':cid,'response_rate':rr,'reply_speed_hours':rs,
            'interview_attendance':ia,'application_completion':ac,'engagement_score':score}

def seed_database(n=500):
    conn = get_conn()
    count = conn.execute('SELECT COUNT(*) FROM candidates').fetchone()[0]
    conn.close()
    if count >= n:
        print(f'✅ Database already has {count} candidates – skipping seed')
        return
    print(f'🌱 Generating {n} candidates...')
    for c in generate_candidates(n):
        insert_candidate(c)
        insert_engagement(gen_engagement(c['candidate_id']))
    print(f'✅ Seeded {n} candidates into SQLite')

print('✅ Data generator ready')

## 🧠 Step 5 – Scoring Engines (Match · Confidence · Engagement)

In [ ]:
def compute_match_score(candidate, jd, semantic_score=0.0):
    cskills = {s.lower() for s in candidate.get('skills',[])}
    jskills = jd.get('skills',[])
    matched  = [s for s in jskills if s.lower() in cskills]
    missing  = [s for s in jskills if s.lower() not in cskills]
    skill_s  = len(matched)/len(jskills) if jskills else 1.0

    exp = candidate.get('experience_years',0)
    emin,emax = jd.get('experience_min',0),jd.get('experience_max',30)
    if emin<=exp<=emax: exp_s=1.0
    elif exp<emin: exp_s=max(0.0,1-(emin-exp)*0.2)
    else: exp_s=max(0.5,1-(exp-emax)*0.05)

    jloc = jd.get('location','')
    loc_s = 1.0 if not jloc or jloc.lower() in candidate.get('location','').lower() else 0.4

    jind = jd.get('industry','')
    ind_s = 1.0 if not jind or jind.lower() in candidate.get('industry','').lower() else 0.6

    ctc_max = jd.get('compensation_max',0)
    ectc    = candidate.get('expected_ctc',0)
    ctc_s   = 1.0 if not ctc_max or ectc<=ctc_max else max(0.3,ctc_max/ectc)

    struct = 0.35*skill_s + 0.25*exp_s + 0.15*loc_s + 0.15*ind_s + 0.10*ctc_s
    sem    = min(1.0, semantic_score/100)
    match  = 0.70*struct + 0.30*sem
    return {
        'match_score': round(match*100,1),
        'skill_score': round(skill_s*100,1),
        'experience_score': round(exp_s*100,1),
        'location_score': round(loc_s*100,1),
        'industry_score': round(ind_s*100,1),
        'structured_score': round(struct*100,1),
        'semantic_score': round(sem*100,1),
        'matched_skills': matched,
        'missing_skills': missing,
    }

def compute_confidence_score(candidate):
    proj_s  = min(1.0, len(candidate.get('projects',[]))/3)
    exp_s   = min(1.0, len(candidate.get('work_experience',[]))/2)
    about_s = min(1.0, len(candidate.get('about_section','').split())/60)
    skill_s = min(1.0, len(candidate.get('skills',[]))/7)
    edu_s   = 1.0 if candidate.get('education') else 0.0
    score   = 0.30*proj_s + 0.25*exp_s + 0.20*about_s + 0.15*skill_s + 0.10*edu_s
    return {'confidence_score': round(score*100,1)}

def compute_engagement_score(cid):
    eng = get_engagement(cid)
    return {'engagement_score': round(eng['engagement_score'],1) if eng else 50.0}

def compute_final_score(m, c, e):
    return round(0.60*m + 0.20*c + 0.20*e, 2)

print('✅ Scoring engines ready')

## 🔍 Step 6 – FAISS Builder & Semantic Search

In [ ]:
import numpy as np, pickle, faiss
from sentence_transformers import SentenceTransformer

FAISS_PATH  = os.environ['FAISS_INDEX_PATH']
EMB_MODEL   = os.environ['EMBEDDING_MODEL']

_embedder       = None
_faiss_index    = None
_meta_store     = []

def get_embedder():
    global _embedder
    if _embedder is None:
        print(f'Loading embedder: {EMB_MODEL}...')
        _embedder = SentenceTransformer(EMB_MODEL)
        print('✅ Embedder loaded')
    return _embedder

def _to_chunks(c):
    cid, name = c['candidate_id'], c.get('name','')
    chunks = []
    if c.get('about_section'):
        chunks.append({'text':f'About {name}: {c["about_section"]}','candidate_id':cid,'chunk_type':'about'})
    for p in c.get('projects',[]):
        chunks.append({'text':f'Project: {p["title"]}. {p["description"]} Skills: {", ".join(p.get("skills_used",[]))}.',
                       'candidate_id':cid,'chunk_type':'project'})
    for w in c.get('work_experience',[]):
        chunks.append({'text':f'{name} worked as {w["title"]} at {w["company"]} for {w["tenure_years"]} years.',
                       'candidate_id':cid,'chunk_type':'experience'})
    chunks.append({'text':f'{name} skills: {", ".join(c.get("skills",[]))}. Industry: {c.get("industry","")}. Role: {c.get("role","")}.',
                   'candidate_id':cid,'chunk_type':'skills'})
    return chunks

def build_faiss_index(candidates):
    global _faiss_index, _meta_store
    os.makedirs(FAISS_PATH, exist_ok=True)
    emb = get_embedder()
    print(f'🔨 Building FAISS index for {len(candidates)} candidates...')
    chunks = []
    for c in candidates: chunks.extend(_to_chunks(c))
    texts  = [ch['text'] for ch in chunks]
    print(f'   Embedding {len(texts)} chunks...')
    vecs   = emb.encode(texts, batch_size=64, show_progress_bar=True,
                        convert_to_numpy=True, normalize_embeddings=True)
    dim    = vecs.shape[1]
    index  = faiss.IndexFlatIP(dim)
    index.add(vecs.astype(np.float32))
    faiss.write_index(index, f'{FAISS_PATH}/index.faiss')
    with open(f'{FAISS_PATH}/metadata.pkl','wb') as f: pickle.dump(chunks,f)
    _faiss_index = index
    _meta_store  = chunks
    print(f'✅ FAISS index built: {index.ntotal} vectors')

def load_faiss_index():
    global _faiss_index, _meta_store
    _faiss_index = faiss.read_index(f'{FAISS_PATH}/index.faiss')
    with open(f'{FAISS_PATH}/metadata.pkl','rb') as f: _meta_store = pickle.load(f)
    print(f'✅ FAISS index loaded: {_faiss_index.ntotal} vectors')

def index_exists():
    return os.path.exists(f'{FAISS_PATH}/index.faiss')

def faiss_search(query, top_k=50):
    global _faiss_index, _meta_store
    if _faiss_index is None: load_faiss_index()
    emb  = get_embedder()
    qvec = emb.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
    scores, idxs = _faiss_index.search(qvec, top_k*4)
    seen, results = set(), []
    for sc, ix in zip(scores[0], idxs[0]):
        if ix<0 or ix>=len(_meta_store): continue
        chunk = _meta_store[ix]
        cid   = chunk['candidate_id']
        results.append({**chunk,'similarity_score':float(sc)})
        seen.add(cid)
        if len(seen) >= top_k: break
    return results

def semantic_search(query, top_k=50):
    chunks = faiss_search(query, top_k=top_k*4)
    scores_map, evidence_map = {}, {}
    for ch in chunks:
        cid = ch['candidate_id']
        sc  = ch['similarity_score']
        if cid not in scores_map or sc > scores_map[cid]: scores_map[cid]=sc
        evidence_map.setdefault(cid,[]).append(ch['text'][:200])
    ranked = sorted(scores_map.items(), key=lambda x:x[1], reverse=True)[:top_k]
    return [{'candidate_id':cid,'semantic_score':round(sc*100,2),
             'evidence_snippets':evidence_map[cid][:3]} for cid,sc in ranked]

def get_rag_context(candidate_id, query, top_n=5):
    chunks = faiss_search(query, top_k=200)
    cchunks = sorted([c for c in chunks if c['candidate_id']==candidate_id],
                      key=lambda x:x['similarity_score'], reverse=True)
    return [c['text'] for c in cchunks[:top_n]]

print('✅ FAISS / Semantic search ready')

## 🤖 Step 7 – LangChain + Groq LLM Chains

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

# Models tried in order — if one is decommissioned, falls back to next
GROQ_MODEL_FALLBACKS = [
    'llama-3.3-70b-versatile',
    'llama-3.1-70b-versatile',
    'llama3-8b-8192',
    'mixtral-8x7b-32768',
    'gemma2-9b-it',
]

_llm = None

def get_llm():
    global _llm
    if _llm is not None:
        return _llm

    api_key = os.environ['GROQ_API_KEY']
    for model in GROQ_MODEL_FALLBACKS:
        try:
            candidate_llm = ChatGroq(model=model, temperature=0, api_key=api_key)
            # Quick probe to confirm model is live
            candidate_llm.invoke('hi')
            _llm = candidate_llm
            print(f'✅ Using Groq model: {model}')
            return _llm
        except Exception as e:
            if 'decommissioned' in str(e).lower() or '400' in str(e):
                print(f'⚠️  {model} decommissioned, trying next...')
                continue
            else:
                # Real error (bad key, network, etc.) — raise immediately
                raise
    raise RuntimeError('❌ All Groq models failed. Check https://console.groq.com/docs/models')

# ── JD Intelligence ───────────────────────────────────────────────────────────
JD_PROMPT = PromptTemplate(input_variables=['jd_text'], template="""
You are an expert recruiter AI. Extract structured information from the job description.
Return ONLY a valid JSON object with these exact keys:
{{"role":"...","experience_min":<num>,"experience_max":<num>,"skills":[...],
  "industry":"...","location":"...","responsibilities":[...],
  "compensation_min":<num>,"compensation_max":<num>,"nice_to_have":[...]}}

Job Description:
{jd_text}
""")

def extract_jd(jd_text):
    llm = get_llm()
    raw = llm.invoke(JD_PROMPT.format(jd_text=jd_text)).content.strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except:
        return {'role':'Unknown','experience_min':0,'experience_max':5,'skills':[],
                'industry':'','location':'','responsibilities':[],'compensation_min':0,
                'compensation_max':0,'nice_to_have':[]}

def format_jd_display(s):
    lines = [f"**Role:** {s.get('role','N/A')}",
             f"**Experience:** {s.get('experience_min',0)}–{s.get('experience_max',0)} years",
             f"**Skills:** {', '.join(s.get('skills',[]))}",
             f"**Industry:** {s.get('industry','N/A')}",
             f"**Location:** {s.get('location','N/A')}"]
    if s.get('compensation_max'):
        lines.append(f"**Compensation:** ₹{s.get('compensation_min',0)}–{s.get('compensation_max',0)} LPA")
    return '\n'.join(lines)

def parse_pdf_jd(file_path):
    from pypdf import PdfReader
    reader = PdfReader(file_path)
    return '\n'.join(p.extract_text() or '' for p in reader.pages)

# ── RAG Match Details ─────────────────────────────────────────────────────────
GAP_PROMPT = PromptTemplate(input_variables=['context','jd_requirements'], template="""
Analyze the candidate profile and identify gaps against job requirements.
Use ONLY information from the profile context below.

Candidate Profile:
{context}

Job Requirements:
{jd_requirements}

Return ONLY valid JSON:
{{"matched":[{{"requirement":"...","evidence":"...","strength":"Strong/Moderate/Weak"}}],
  "missing":[{{"requirement":"...","impact":"High/Medium/Low"}}],
  "overall_gap_summary":"...","match_percentage":<0-100>}}
""")

def get_match_details(cid, structured_jd, candidate):
    jd_req = '\n'.join([
        f"- Role: {structured_jd.get('role','')}",
        f"- Experience: {structured_jd.get('experience_min',0)}-{structured_jd.get('experience_max',0)} years",
        f"- Skills: {', '.join(structured_jd.get('skills',[]))}",
    ] + [f"- {r}" for r in structured_jd.get('responsibilities',[])[:4]])

    query  = f"{structured_jd.get('role','')} {' '.join(structured_jd.get('skills',[]))}"
    chunks = get_rag_context(cid, query, top_n=5)
    if not chunks:
        c = candidate
        chunks = [f"Name:{c['name']}, Skills:{c['skills']}, About:{c.get('about_section','')[:300]}"]
        for p in c.get('projects',[])[:2]: chunks.append(f"Project – {p['title']}: {p['description']}")
    context = '\n\n'.join(chunks)

    llm = get_llm()
    raw = llm.invoke(GAP_PROMPT.format(context=context, jd_requirements=jd_req)).content.strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        gap = json.loads(raw.strip())
    except:
        gap = {'matched':[],'missing':[],'overall_gap_summary':'Parse error.','match_percentage':0}
    return {'gap_analysis':gap,'context_chunks':chunks[:3]}

# ── Question Generation ───────────────────────────────────────────────────────
Q_PROMPT = PromptTemplate(input_variables=['missing','role','context'], template="""
Generate 3 targeted screening questions for a {role} candidate.
Missing requirements: {missing}
Candidate context: {context}

Return ONLY valid JSON array:
[{{"question":"...","targets_requirement":"...","potential_score_impact":"+X%"}}]
""")

A_PROMPT = PromptTemplate(input_variables=['question','answer','role','requirement'], template="""
Evaluate this candidate answer for a {role} position.
Requirement: {requirement}
Question: {question}
Answer: {answer}

Return ONLY valid JSON:
{{"assessment_score":<0-100>,"score_impact":<-5 to 10>,
  "verdict":"Strong/Moderate/Weak","feedback":"one sentence"}}
""")

def generate_questions(missing_skills, role, context):
    llm = get_llm()
    raw = llm.invoke(Q_PROMPT.format(
        missing='\n'.join(f'- {s}' for s in missing_skills),
        role=role, context=context[:600])).content.strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except:
        return [{'question':f'Can you describe your experience with {s}?',
                 'targets_requirement':s,'potential_score_impact':'+3%'} for s in missing_skills[:3]]

def assess_answer(question, answer, role, requirement):
    llm = get_llm()
    raw = llm.invoke(A_PROMPT.format(question=question, answer=answer,
                                      role=role, requirement=requirement)).content.strip()
    if '```' in raw:
        raw = raw.split('```')[1]
        if raw.startswith('json'): raw = raw[4:]
    try:
        return json.loads(raw.strip())
    except:
        return {'assessment_score':60,'score_impact':2,'verdict':'Moderate','feedback':'Answer recorded.'}

print('✅ LangChain + Groq chains ready (auto-fallback enabled)')

## 🔗 Step 8 – Hybrid Search Pipeline

In [ ]:
def structured_filter(skills=None, location='', experience_min=0, experience_max=30,
                      industry='', limit=300):
    conn = get_conn()
    params, clauses = [], []
    if experience_min>0:  clauses.append('experience_years>=?'); params.append(experience_min)
    if experience_max<30: clauses.append('experience_years<=?'); params.append(experience_max)
    if location:          clauses.append('LOWER(location) LIKE ?'); params.append(f'%{location.lower()}%')
    if industry:          clauses.append('LOWER(industry) LIKE ?'); params.append(f'%{industry.lower()}%')
    for s in (skills or []):
        clauses.append('LOWER(skills) LIKE ?'); params.append(f'%{s.lower()}%')
    where = f'WHERE {" AND ".join(clauses)}' if clauses else ''
    rows  = conn.execute(f'SELECT * FROM candidates {where} LIMIT {limit}', params).fetchall()
    conn.close()
    return [_parse(r) for r in rows]

def run_hybrid_search(structured_jd, top_k=20):
    job_id = structured_jd.get('job_id', f'JOB-{str(uuid.uuid4())[:6].upper()}')

    # Step 1: SQL hard filter (location, exp, industry, skills)
    print('Step 1: SQL structured filter...')
    sql_candidates = structured_filter(
        skills=structured_jd.get('skills', [])[:5],
        location=structured_jd.get('location', ''),
        experience_min=structured_jd.get('experience_min', 0),
        experience_max=structured_jd.get('experience_max', 30),
        industry=structured_jd.get('industry', ''),
    )
    sql_ids = {c['candidate_id'] for c in sql_candidates}
    sql_map  = {c['candidate_id']: c for c in sql_candidates}
    print(f'   {len(sql_ids)} candidates passed SQL filter')

    if not sql_ids:
        print('   No SQL matches.')
        return job_id, []

    # Step 2: FAISS semantic re-rank WITHIN the SQL pool only
    # FAISS never adds new candidates - it only reorders what SQL approved
    print('Step 2: FAISS semantic re-rank within SQL pool...')
    query = ' '.join(filter(None, [
        structured_jd.get('role', ''),
        ', '.join(structured_jd.get('skills', [])[:6]),
        '. '.join(structured_jd.get('responsibilities', [])[:2]),
    ]))
    sem_results = semantic_search(query, top_k=500)
    sem_map = {
        r['candidate_id']: r
        for r in sem_results
        if r['candidate_id'] in sql_ids
    }
    print(f'   {len(sem_map)} of {len(sql_ids)} SQL candidates matched semantically')

    # Step 3: Score every SQL candidate
    print('Step 3: Scoring...')
    ranked = []
    for cid, candidate in sql_map.items():
        sem_sc = sem_map.get(cid, {}).get('semantic_score', 0.0)
        m  = compute_match_score(candidate, structured_jd, sem_sc)
        co = compute_confidence_score(candidate)
        e  = compute_engagement_score(cid)
        fs = compute_final_score(m['match_score'], co['confidence_score'], e['engagement_score'])
        ranked.append({
            **candidate, **m,
            'confidence_score': co['confidence_score'],
            'engagement_score': e['engagement_score'],
            'final_score': fs,
            'updated_score': fs,
            'semantic_evidence': sem_map.get(cid, {}).get('evidence_snippets', []),
        })

    ranked.sort(key=lambda x: x['final_score'], reverse=True)
    top = ranked[:top_k]

    save_job(job_id, structured_jd.get('_raw_jd', ''), structured_jd)
    for c in top:
        upsert_job_candidate(job_id, c['candidate_id'], {
            'match_score':      c['match_score'],
            'confidence_score': c['confidence_score'],
            'engagement_score': c['engagement_score'],
            'final_score':      c['final_score'],
        })

    print(f'Done. Top {len(top)} from {len(sql_ids)} SQL-qualified candidates.')
    return job_id, top

print('Hybrid search pipeline ready')

## 📋 Step 9 – Workflow, Memory & Submission

In [ ]:
VALID_STATUSES = ['Recommended','Contacted','Responded','Screening Complete',
                  'Interview Scheduled','Rejected','Selected','Offered','Hired']

def move_to_status(job_id, cid, status, notes=''):
    update_candidate_status(job_id, cid, status, notes)

def get_workflow_history(job_id, cid):
    conn = get_conn()
    rows = conn.execute('SELECT * FROM workflow_history WHERE job_id=? AND candidate_id=? ORDER BY created_at',(job_id,cid)).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def get_pipeline_summary(job_id):
    cands = get_job_candidates(job_id)
    summary = {s:0 for s in VALID_STATUSES}
    for c in cands: summary[c.get('status','Recommended')] += 1
    return {k:v for k,v in summary.items() if v>0}

def get_assessments_for_report(job_id, cid):
    conn = get_conn()
    rows = conn.execute(
        'SELECT question,answer,assessment_score,score_impact,verdict,feedback,targets_requirement FROM job_assessments WHERE job_id=? AND candidate_id=? ORDER BY created_at',
        (job_id,cid)).fetchall()
    conn.close()
    return [dict(r) for r in rows]

def apply_assessment_rerank(job_id, cid, question, answer, result, current_score, requirement=''):
    impact    = result.get('score_impact', 0)
    new_score = min(100.0, max(0.0, current_score + impact))
    save_assessment(job_id, cid, question, answer,
                    result.get('assessment_score', 0), impact,
                    verdict=result.get('verdict',''),
                    feedback=result.get('feedback',''),
                    targets_requirement=requirement)
    update_assessment_score(job_id, cid, new_score)
    return round(new_score, 2)

def generate_report(job_id, cid):
    candidate   = get_candidate(cid)
    job         = get_job(job_id)
    conn        = get_conn()
    scores_row  = conn.execute('SELECT * FROM job_candidates WHERE job_id=? AND candidate_id=?',(job_id,cid)).fetchone()
    conn.close()
    scores      = dict(scores_row) if scores_row else {}
    memory      = get_recruiter_memory(job_id, cid)
    assessments = get_assessments_for_report(job_id, cid)
    jd          = job.get('structured_jd',{}) if job else {}
    req_s       = jd.get('skills',[])
    cand_s      = candidate.get('skills',[])
    matched     = [s for s in req_s if s.lower() in {x.lower() for x in cand_s}]
    missing     = [s for s in req_s if s.lower() not in {x.lower() for x in cand_s}]

    match_evidence, context_chunks = [], []
    try:
        details = get_match_details(cid, jd, candidate)
        gap     = details.get('gap_analysis',{})
        match_evidence = [
            {'requirement':m.get('requirement',''),'evidence':m.get('evidence',''),'strength':m.get('strength','')}
            for m in gap.get('matched',[])
        ]
        context_chunks = details.get('context_chunks',[])[:4]
    except Exception:
        pass

    recruiter_notes = list({m.get('recruiter_notes','') for m in memory if m.get('recruiter_notes','').strip()})
    assessed_qa = [
        {'question':a['question'],'answer':a['answer'],
         'targets':a.get('targets_requirement',''),
         'verdict':a.get('verdict',''),'feedback':a.get('feedback',''),
         'assessment_score':a.get('assessment_score',0),'score_impact':a.get('score_impact',0)}
        for a in assessments
    ]
    final_display = scores.get('updated_score') or scores.get('final_score',0)
    total_uplift  = sum(a.get('score_impact',0) for a in assessments)

    assessed_questions = {a['question'] for a in assessments}
    clarifications = [
        {'question': m.get('question',''), 'answer': m.get('answer','')}
        for m in memory
        if m.get('question','').strip() and m.get('answer','').strip()
        and m['question'] not in assessed_questions
    ]

    return {
        'report_id': f'RPT-{job_id}-{cid}',
        'generated_at': datetime.now().isoformat(),
        'candidate': candidate, 'job': jd,
        'scores': {**scores, 'displayed_final_score': final_display, 'assessment_uplift': round(total_uplift,1)},
        'matched_skills': matched, 'missing_skills': missing,
        'match_evidence': match_evidence, 'context_chunks': context_chunks,
        'projects': candidate.get('projects',[])[:3],
        'recruiter_notes': recruiter_notes,
        'assessed_qa': assessed_qa,
        'clarifications': clarifications,
    }

def format_report(r):
    c, s   = r['candidate'], r['scores']
    final  = s.get('displayed_final_score') or s.get('final_score',0)
    uplift = s.get('assessment_uplift',0)
    lines  = [
        '='*62, '  CANDIDATE SUBMISSION REPORT',
        f"  {r['report_id']}  |  {r['generated_at'][:16]}", '='*62, '',
        f"CANDIDATE  : {c.get('name')}",
        f"Location   : {c.get('location')}   Exp: {c.get('experience_years')} yrs",
        f"Industry   : {c.get('industry')}",
        f"Expected CTC: {c.get('expected_ctc')} LPA", '',
        f"ROLE: {r['job'].get('role','N/A')}", '',
        '-'*62, 'SCORES', '-'*62,
        f"  Match Score      : {s.get('match_score',0):.1f}%",
        f"  Confidence Score : {s.get('confidence_score',0):.1f}%",
        f"  Engagement Score : {s.get('engagement_score',0):.1f}%",
        f"  Base Final Score : {s.get('final_score',0):.1f}%",
        f"  Assessment Uplift: {uplift:+.1f} pts  ({len(r.get('assessed_qa',[]))} question(s))",
        f"  FINAL SCORE      : {final:.1f}%", '',
    ]
    lines += ['-'*62, 'MATCHING EVIDENCE (AI-retrieved from profile)', '-'*62]
    if r.get('match_evidence'):
        for ev in r['match_evidence']:
            lines.append(f"  {ev['requirement']}")
            if ev.get('evidence'): lines.append(f"     Evidence : {ev['evidence']}")
            if ev.get('strength'): lines.append(f"     Strength : {ev['strength']}")
    else:
        for sk in r.get('matched_skills',[]): lines.append(f'  + {sk}')
    if r.get('missing_skills'):
        lines += ['', '  GAPS:']
        for sk in r['missing_skills']: lines.append(f'  - {sk}')
    lines.append('')
    if r.get('context_chunks'):
        lines += ['-'*62, 'RETRIEVED PROFILE SECTIONS (RAG Context)', '-'*62]
        for i,chunk in enumerate(r['context_chunks'],1): lines.append(f'  [{i}] {chunk[:300]}')
        lines.append('')
    if r.get('projects'):
        lines += ['-'*62, 'KEY PROJECTS', '-'*62]
        for p in r['projects']:
            lines += [f"  {p.get('title','')}", f"  {p.get('description','')[:200]}",
                      f"  Skills: {', '.join(p.get('skills_used',[]))}  Impact: {p.get('impact','')}", '']
    if r.get('clarifications'):
        lines += ['-'*62, 'RECRUITER CLARIFICATIONS (from Match Details)', '-'*62]
        for i,cl in enumerate(r['clarifications'],1):
            lines += [
                f"  Q{i}: {cl['question']}",
                f"  A{i}: {cl['answer'][:400]}", ''
            ]
    if r.get('assessed_qa'):
        lines += ['-'*62, 'SCREENING Q&A (with AI Assessment)', '-'*62]
        for i,qa in enumerate(r['assessed_qa'],1):
            lines += [
                f"  Q{i}: {qa['question']}",
                f"  Targets: {qa.get('targets','')}",
                f"  Answer : {qa['answer'][:300]}",
                f"  Verdict: {qa.get('verdict','N/A')}  Score: {qa.get('assessment_score',0):.0f}/100  Impact: {qa.get('score_impact',0):+.1f} pts",
            ]
            if qa.get('feedback'): lines.append(f"  AI Feedback: {qa['feedback']}")
            lines.append('')
    if r.get('recruiter_notes'):
        lines += ['-'*62, 'RECRUITER NOTES', '-'*62]
        for n in r['recruiter_notes']: lines.append(f'  * {n}')
        lines.append('')
    lines += ['='*62, 'Generated by AI Recruiter Copilot - TalentXO']
    return '\n'.join(lines)

def export_pdf(report):
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
    from reportlab.lib.units import cm
    os.makedirs('exports', exist_ok=True)
    path = f"exports/submission_{report['report_id']}.pdf"
    doc  = SimpleDocTemplate(path, pagesize=A4, leftMargin=2*cm, rightMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
    styles = getSampleStyleSheet()
    h1 = ParagraphStyle('H1', parent=styles['Heading1'], textColor=colors.HexColor('#1a1a2e'), fontSize=18)
    h2 = ParagraphStyle('H2', parent=styles['Heading2'], textColor=colors.HexColor('#16213e'), fontSize=13)
    sm = ParagraphStyle('SM', parent=styles['Normal'],   textColor=colors.HexColor('#4a4a6a'), fontSize=9)
    bd = ParagraphStyle('BD', parent=styles['Normal'],   fontSize=10, leading=14)
    story = []
    c = report['candidate']; s = report['scores']
    final = s.get('displayed_final_score') or s.get('final_score',0)
    uplift = s.get('assessment_uplift',0)
    story.append(Paragraph('Candidate Submission Report', h1))
    story.append(Paragraph(f"Generated: {report['generated_at'][:16]}  |  {report['report_id']}", sm))
    story.append(HRFlowable(width='100%', thickness=1, color=colors.HexColor('#e0e0e0')))
    story.append(Spacer(1, 0.3*cm))
    story.append(Paragraph('Candidate Overview', h2))
    t = Table([
        ['Name', c.get('name','')], ['Location', c.get('location','')],
        ['Experience', f"{c.get('experience_years',0)} years"], ['Industry', c.get('industry','')],
        ['Expected CTC', f"Rs {c.get('expected_ctc',0):.1f} LPA"],
    ], colWidths=[4*cm,13*cm])
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(0,-1),colors.HexColor('#f0f4ff')),
                            ('FONTNAME',(0,0),(0,-1),'Helvetica-Bold'),
                            ('FONTSIZE',(0,0),(-1,-1),9),
                            ('GRID',(0,0),(-1,-1),0.5,colors.HexColor('#d0d0d0'))]))
    story.append(t); story.append(Spacer(1,0.4*cm))
    story.append(Paragraph('Match Scores', h2))
    st = Table([
        [f"Match: {s.get('match_score',0):.1f}%", f"Confidence: {s.get('confidence_score',0):.1f}%",
         f"Engagement: {s.get('engagement_score',0):.1f}%", f"Uplift: {uplift:+.1f}pts"],
        [f"Base Score: {s.get('final_score',0):.1f}%", '', '', f"FINAL: {final:.1f}%"],
    ])
    st.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,-1),colors.HexColor('#e8f5e9')),
                             ('FONTNAME',(0,0),(-1,-1),'Helvetica-Bold'),
                             ('FONTSIZE',(0,0),(-1,-1),10),
                             ('BACKGROUND',(3,1),(3,1),colors.HexColor('#a5d6a7'))]))
    story.append(st); story.append(Spacer(1,0.4*cm))
    story.append(Paragraph('Matching Evidence (AI-retrieved from profile)', h2))
    if report.get('match_evidence'):
        for ev in report['match_evidence']:
            story.append(Paragraph(f"<b>{ev['requirement']}</b>", bd))
            if ev.get('evidence'): story.append(Paragraph(f"Evidence: {ev['evidence']}", sm))
            if ev.get('strength'): story.append(Paragraph(f"Strength: {ev['strength']}", sm))
            story.append(Spacer(1,0.1*cm))
    else:
        for sk in report.get('matched_skills',[]): story.append(Paragraph(f'+ {sk}', bd))
    if report.get('missing_skills'):
        story.append(Paragraph('Gaps:', bd))
        for sk in report.get('missing_skills',[]): story.append(Paragraph(f'- {sk}', bd))
    story.append(Spacer(1,0.4*cm))
    if report.get('context_chunks'):
        story.append(Paragraph('Retrieved Profile Sections (RAG Context)', h2))
        for i,chunk in enumerate(report['context_chunks'],1):
            story.append(Paragraph(f'<b>[{i}]</b> {chunk[:350]}', bd))
            story.append(Spacer(1,0.15*cm))
        story.append(Spacer(1,0.3*cm))
    if report.get('projects'):
        story.append(Paragraph('Key Projects', h2))
        for p in report['projects']:
            story.append(Paragraph(f"<b>{p.get('title','')}</b>", bd))
            story.append(Paragraph(p.get('description',''), bd))
            story.append(Paragraph(f"Skills: {', '.join(p.get('skills_used',[]))} | Impact: {p.get('impact','')}", sm))
            story.append(Spacer(1,0.2*cm))
    if report.get('assessed_qa'):
        story.append(Paragraph('Screening Q&amp;A with AI Assessment', h2))
        for i,qa in enumerate(report['assessed_qa'],1):
            story.append(Paragraph(f"<b>Q{i}: {qa['question']}</b>  [{qa.get('targets','')}]", bd))
            story.append(Paragraph(f"Answer: {qa['answer'][:300]}", bd))
            story.append(Paragraph(
                f"Verdict: <b>{qa.get('verdict','N/A')}</b>  Score: {qa.get('assessment_score',0):.0f}/100  Impact: {qa.get('score_impact',0):+.1f}pts", bd))
            if qa.get('feedback'): story.append(Paragraph(f"<i>AI Feedback: {qa['feedback']}</i>", sm))
            story.append(Spacer(1,0.25*cm))
    if report.get('recruiter_notes'):
        story.append(Paragraph('Recruiter Notes', h2))
        for note in report['recruiter_notes']:
            story.append(Paragraph(f'* {note}', bd))
        story.append(Spacer(1,0.3*cm))
    story.append(HRFlowable(width='100%',thickness=1,color=colors.HexColor('#e0e0e0')))
    story.append(Paragraph('Generated by AI Recruiter Copilot - TalentXO', sm))
    doc.build(story)
    print(f'PDF exported: {path}')
    return path

def record_outcome(job_id, cid, outcome, notes=''):
    save_feedback(job_id, cid, outcome, notes)

print('Workflow, memory, submission ready')


## 🚀 Step 10 – Initialize System (DB + Seed + FAISS)

In [ ]:
init_db()
seed_database(500)

if index_exists():
    print('FAISS index found – loading...')
    load_faiss_index()
else:
    print('Building FAISS index (~2-3 min)...')
    build_faiss_index(get_all_candidates())

print('\n🎉 System fully initialized!')

## 🧪 Step 11 – Demo: JD Intelligence Engine

In [ ]:
SAMPLE_JD = """
We are hiring a Product Analyst for our SaaS platform in Bangalore.

Requirements:
- 2-4 years of product analytics or business analytics experience
- Strong SQL skills for data extraction and analysis
- Experience with dashboarding tools (Tableau, Power BI)
- Stakeholder management and communication skills
- A/B testing and experimentation knowledge
- Python for data analysis (nice to have)
- Leadership experience managing junior analysts (nice to have)

Compensation: 12-20 LPA. Location: Bangalore (Hybrid)
"""

print('🧠 Extracting JD structure via Groq Llama 3 70B...')
structured_jd = extract_jd(SAMPLE_JD)
structured_jd['job_id']  = f'JOB-{str(uuid.uuid4())[:6].upper()}'
structured_jd['_raw_jd'] = SAMPLE_JD

print('\nExtracted JD:')
print(json.dumps({k:v for k,v in structured_jd.items() if not k.startswith('_')}, indent=2))

## 🔍 Step 12 – Demo: Hybrid Search (SQL + Semantic + Ranking)

In [ ]:
import pandas as pd

job_id, ranked_candidates = run_hybrid_search(structured_jd, top_k=10)

rows = []
for rank, c in enumerate(ranked_candidates, 1):
    rows.append({'Rank':rank,'Name':c['name'],'Location':c['location'],
                 'Exp':c['experience_years'],'Skills':', '.join(c['skills'][:3]),
                 'Match%':f"{c['match_score']:.1f}",'Confidence%':f"{c['confidence_score']:.1f}",
                 'Engagement%':f"{c['engagement_score']:.1f}",'Final%':f"{c['final_score']:.1f}"})
df = pd.DataFrame(rows)
print(f'\nJob ID: {job_id}  |  Top {len(ranked_candidates)} Candidates:\n')
df

## 🎯 Step 13 – Demo: RAG Match Details + Gap Detection

In [ ]:
top = ranked_candidates[0]
cid = top['candidate_id']
candidate = get_candidate(cid)

print(f'Analyzing: {candidate["name"]} ({cid})')
print(f'Skills: {candidate["skills"]}')
print(f'Current Score: {top["final_score"]:.1f}%\n')

details = get_match_details(cid, structured_jd, candidate)
gap     = details['gap_analysis']

print('✅ MATCHED REQUIREMENTS:')
for m in gap.get('matched',[]):
    print(f"  ✅ {m['requirement']} — {m.get('strength','')} [{m.get('evidence','')}]")

print('\n❌ GAPS / MISSING:')
for m in gap.get('missing',[]):
    print(f"  ❌ {m['requirement']} — Impact: {m.get('impact','')}")

print(f"\n📊 Overall: {gap.get('overall_gap_summary','')}")
print(f'Match: {gap.get("match_percentage",0)}%')

## 💬 Step 14 – Demo: AI Question Generation

In [ ]:
missing_requirements = [m['requirement'] for m in gap.get('missing',[])][:3]
if not missing_requirements:
    missing_requirements = ['Leadership experience','Team management']

context = candidate.get('about_section','') + ' '.join(
    p.get('description','') for p in candidate.get('projects',[])[:2])

questions = generate_questions(missing_requirements, structured_jd.get('role',''), context)

print(f'Generated Screening Questions for {candidate["name"]}:\n')
for i, q in enumerate(questions, 1):
    print(f'Q{i}: {q["question"]}')
    print(f'   Targets: {q.get("targets_requirement","")}')
    print(f'   Potential Impact: {q.get("potential_score_impact","")}\n')

## 📝 Step 15 – Demo: Candidate Assessment & Dynamic Re-Ranking

In [ ]:
question = questions[0]['question'] if questions else 'Have you managed a team?'
answer   = """
Yes, I have managed a team of 4 junior analysts for the past 18 months.
I ran weekly sprint planning, conducted performance reviews, and mentored them 
on SQL and data visualization skills. I also worked directly with product managers 
and C-suite stakeholders to present quarterly analytics findings.
"""

print(f'Question: {question}')
print(f'Answer: {answer[:150]}...\n')

result = assess_answer(
    question=question, answer=answer,
    role=structured_jd.get('role','Product Analyst'),
    requirement=questions[0].get('targets_requirement','Leadership') if questions else 'Leadership'
)
print('Assessment Result:')
print(f'  Verdict:      {result["verdict"]}')
print(f'  Score:        {result["assessment_score"]}/100')
print(f'  Score Impact: +{result["score_impact"]} points')
print(f'  Feedback:     {result["feedback"]}')

new_score = apply_assessment_rerank(job_id, cid, question, answer, result, top['final_score'])
print(f'\n🔄 Dynamic Re-Ranking (this job only):')
print(f'  Before: {top["final_score"]:.1f}%')
print(f'  After:  {new_score:.1f}%')
print('  (Master profile unchanged)')

## 📋 Step 16 – Demo: Workflow Tracker

In [ ]:
# Move candidate through pipeline
move_to_status(job_id, cid, 'Contacted',          'Sent outreach email')
move_to_status(job_id, cid, 'Responded',          'Candidate confirmed interest')
move_to_status(job_id, cid, 'Screening Complete', 'Strong screening call')

# Save recruiter memory
save_recruiter_memory(job_id, cid, question, answer, 'Strong leadership. Ready for client interview.')

print(f'Pipeline Summary for {job_id}:')
for status, count in get_pipeline_summary(job_id).items():
    print(f'  {status}: {count}')

print(f'\nWorkflow History for {candidate["name"]}:')
for h in get_workflow_history(job_id, cid):
    print(f'  [{h["created_at"][:16]}] {h["status"]} — {h.get("notes","")}')

## 📤 Step 17 – Demo: One-Click Submission Report + PDF

In [ ]:
report = generate_report(job_id, cid)
print(format_report(report))

In [ ]:
# Export to PDF
pdf_path = export_pdf(report)

# Download in Colab
from google.colab import files
files.download(pdf_path)

## 📊 Step 18 – Demo: Feedback Loop & Analytics

In [ ]:
import plotly.graph_objects as go

# Record outcomes
record_outcome(job_id, cid,                           'Interviewed', 'Strong technical skills')
record_outcome(job_id, ranked_candidates[1]['candidate_id'], 'Rejected',   'Not enough experience')
record_outcome(job_id, ranked_candidates[2]['candidate_id'], 'Hired',      'Best fit overall')
record_outcome(job_id, ranked_candidates[3]['candidate_id'], 'Offered',    'Offer extended')
record_outcome(job_id, ranked_candidates[4]['candidate_id'], 'Selected',   'Cleared all rounds')

data   = get_feedback_analytics()
labels = [d['outcome'] for d in data]
values = [d['count'] for d in data]
colors = {'Hired':'#4caf50','Offered':'#8bc34a','Selected':'#03a9f4',
           'Interviewed':'#ff9800','Rejected':'#f44336'}

fig = go.Figure(go.Pie(
    labels=labels, values=values,
    marker_colors=[colors.get(l,'#9e9e9e') for l in labels],
    hole=0.4, textinfo='label+percent+value'
))
fig.update_layout(title='Hiring Outcome Distribution', height=400)
fig.show()

conn = get_conn()
total     = conn.execute('SELECT COUNT(*) FROM candidates').fetchone()[0]
contacted = conn.execute("SELECT COUNT(DISTINCT candidate_id) FROM job_candidates WHERE status!='Recommended'").fetchone()[0]
hired     = conn.execute("SELECT COUNT(*) FROM feedback WHERE outcome='Hired'").fetchone()[0]
conn.close()

print(f'\n📊 Reutilization Metrics')
print(f'  Total Candidates:    {total}')
print(f'  Candidates Engaged:  {contacted}')
print(f'  Reutilization Rate:  {contacted/total*100:.1f}%')
print(f'  Total Hired:         {hired}')

## 🌐 Step 19 – Launch Gradio UI (All 8 Screens)

In [ ]:
import gradio as gr
import pandas as pd

_app_state = {
    'job_id': None, 'structured_jd': None,
    'ranked_candidates': [], 'generated_questions': [], 'submission_report': None
}

THEME = gr.themes.Soft(primary_hue='blue', secondary_hue='indigo')

# ── helpers ───────────────────────────────────────────────────────────────────

def _candidate_choices():
    return [f"{c['name']} ({c['candidate_id']})" for c in _app_state.get('ranked_candidates', [])]

def _cid(sel):
    return sel.split('(')[-1].rstrip(')')

def _get_current_score(job_id, candidate_id):
    """Always reads updated_score (post-assessment) from DB, not the original final_score."""
    conn = get_conn()
    row  = conn.execute(
        'SELECT updated_score, final_score FROM job_candidates WHERE job_id=? AND candidate_id=?',
        (job_id, candidate_id)
    ).fetchone()
    conn.close()
    if not row:
        return 70.0
    # updated_score starts equal to final_score; grows with each assessment
    return row['updated_score']

def _question_already_answered(job_id, candidate_id, question):
    """Prevent the same question from being scored twice."""
    conn = get_conn()
    exists = conn.execute(
        'SELECT 1 FROM job_assessments WHERE job_id=? AND candidate_id=? AND question=?',
        (job_id, candidate_id, question)
    ).fetchone()
    conn.close()
    return exists is not None


def process_voice_jd(audio):
    if audio is None:
        return '❌ No audio recorded.', '', gr.update(visible=False)
    try:
        import speech_recognition as sr
        r = sr.Recognizer()
        with sr.AudioFile(audio) as src_:
            audio_data = r.record(src_)
        text = r.recognize_google(audio_data)
    except Exception as e:
        return f'❌ Transcription failed: {e}', '', gr.update(visible=False)
    return app_process_jd(text)

def run_freetext_search(query):
    if not query.strip():
        return '❌ Enter a search query.', None, gr.update(choices=[])
    s = extract_jd(query)
    s['_raw_jd'] = query
    s['job_id'] = f'JOB-{str(uuid.uuid4())[:6].upper()}'
    _app_state['structured_jd'] = s
    return app_run_search()

def ask_clarification(sel, question):
    if not sel or not question.strip():
        return '❌ Select a candidate and enter a question.'
    cid = _cid(sel)
    c = get_candidate(cid)
    jd = _app_state.get('structured_jd', {})
    jid = _app_state.get('job_id', '')
    ans = answer_recruiter_question(cid, question, jd, c)
    save_recruiter_memory(jid, cid, question, ans, '')
    return f'**A:** {ans}'

# ── Tab 1: JD Upload ──────────────────────────────────────────────────────────

def app_process_jd(jd_text):
    if not jd_text.strip():
        return '❌ Enter a job description.', '', gr.update(visible=False)
    s = extract_jd(jd_text)
    s['_raw_jd'] = jd_text
    s['job_id']  = f'JOB-{str(uuid.uuid4())[:6].upper()}'
    _app_state['structured_jd'] = s
    return f"✅ JD parsed! Job ID: {s['job_id']}", format_jd_display(s), gr.update(visible=True)

# ── Tab 2: Search ─────────────────────────────────────────────────────────────

def app_run_search():
    if not _app_state.get('structured_jd'):
        return '❌ Parse a JD first.', None, gr.update(choices=[])
    jid, candidates = run_hybrid_search(_app_state['structured_jd'], top_k=20)
    _app_state['job_id'] = jid
    _app_state['ranked_candidates'] = candidates
    rows = [{
        'Rank': i+1, 'Name': c['name'], 'Location': c['location'],
        'Exp': c['experience_years'], 'Skills': ', '.join(c['skills'][:3]),
        'Match%': f"{c['match_score']:.1f}",
        'Confidence%': f"{c['confidence_score']:.1f}",
        'Engagement%': f"{c['engagement_score']:.1f}",
        'Final%': f"{c['final_score']:.1f}",
        'ID': c['candidate_id'],
    } for i, c in enumerate(candidates)]
    return (
        f"✅ Found {len(candidates)} candidates for {jid}",
        pd.DataFrame(rows),
        gr.update(choices=_candidate_choices())
    )

# ── Tab 3: Match Details (RAG) ────────────────────────────────────────────────

def app_match_details(sel):
    """
    Retrieves the top FAISS chunks for this candidate vs the JD,
    sends ONLY those chunks to the LLM (no hallucination),
    returns matched requirements with evidence text + missing gaps.
    """
    if not sel:
        return 'Select a candidate.', '', '', ''
    cid  = _cid(sel)
    c    = get_candidate(cid)
    jd   = _app_state.get('structured_jd', {})
    d    = get_match_details(cid, jd, c)
    gap  = d.get('gap_analysis', {})

    matched_text = '\n'.join(
        f"✅  {m['requirement']}\n     Evidence: {m.get('evidence','')}\n     Strength: {m.get('strength','')}"
        for m in gap.get('matched', [])
    ) or 'No matched requirements found.'

    missing_text = '\n'.join(
        f"❌  {m['requirement']}  [Impact: {m.get('impact','')}]"
        for m in gap.get('missing', [])
    ) or 'No gaps — strong match!'

    # Show the actual text chunks retrieved from the vector store
    chunks_text = '\n\n── chunk ──\n'.join(d.get('context_chunks', [])[:4])

    return matched_text, missing_text, gap.get('overall_gap_summary', ''), chunks_text

# ── Tab 4: Assessment ─────────────────────────────────────────────────────────

def app_gen_questions(sel):
    if not sel:
        return 'Select a candidate.', gr.update(choices=[])
    cid  = _cid(sel)
    c    = get_candidate(cid)
    jd   = _app_state.get('structured_jd', {})
    req  = jd.get('skills', [])
    miss = [s for s in req if s.lower() not in {x.lower() for x in c.get('skills', [])}][:3]
    if not miss:
        miss = req[:2]
    ctx = c.get('about_section', '') + ' '.join(
        p.get('description', '') for p in c.get('projects', [])[:2])
    qs  = generate_questions(miss, jd.get('role', ''), ctx)
    _app_state['generated_questions'] = qs
    display = '\n\n'.join(
        f"**Q{i+1}: {q['question']}**\n"
        f"   Targets: {q.get('targets_requirement','')}\n"
        f"   Potential impact: {q.get('potential_score_impact','')}"
        for i, q in enumerate(qs)
    )
    return display, gr.update(choices=[q['question'] for q in qs])

def app_assess(sel, question, answer):
    """
    - Reads updated_score (not original final_score) so Q2 builds on Q1
    - One-time guard: same question cannot be scored twice
    - Score is capped at 100
    """
    if not sel or not question or not answer:
        return '❌ Select candidate, pick a question, and enter an answer.', '', ''

    cid = _cid(sel)
    jid = _app_state.get('job_id', '')
    jd  = _app_state.get('structured_jd', {})

    # ── One-time guard ────────────────────────────────────────────────────────
    if _question_already_answered(jid, cid, question):
        cur = _get_current_score(jid, cid)
        return (
            f'⚠️  This question was already assessed. Current score: {cur:.1f}%',
            'Question already scored — score not changed.',
            f'Score remains: **{cur:.1f}%**'
        )

    req    = next((q.get('targets_requirement', 'General')
                   for q in _app_state.get('generated_questions', [])
                   if q['question'] == question), 'General')

    result = assess_answer(question, answer, jd.get('role', ''), req)

    # ── Build on current updated_score, not original final_score ─────────────
    cur_score = _get_current_score(jid, cid)
    impact    = result.get('score_impact', 0)

    # Cap: score cannot exceed 100
    new_score = min(100.0, max(0.0, cur_score + impact))

    # Persist assessment + updated score
    save_assessment(jid, cid, question, answer,
                    result.get('assessment_score', 0), impact,
                    verdict=result.get('verdict', ''),
                    feedback=result.get('feedback', ''),
                    targets_requirement=req)
    update_assessment_score(jid, cid, new_score)

    return (
        f"✅ Assessed. Score: {cur_score:.1f}% → {new_score:.1f}%",
        (f"**Verdict: {result['verdict']}**\n\n"
         f"Assessment Score: {result['assessment_score']}/100\n"
         f"Score Impact: {impact:+.1f} pts\n\n"
         f"_{result['feedback']}_"),
        f"{cur_score:.1f}% → **{new_score:.1f}%** (capped at 100)"
    )

# ── Tab 5: Workflow ───────────────────────────────────────────────────────────

def app_update_status(sel, status, notes):
    if not sel or not status:
        return '❌ Select a candidate and a status.', None
    cid  = _cid(sel)
    jid  = _app_state.get('job_id', '')
    move_to_status(jid, cid, status, notes)
    hist = get_workflow_history(jid, cid)
    return (
        f'✅ Status updated → {status}',
        pd.DataFrame([{
            'Status': h['status'],
            'Notes': h.get('notes', ''),
            'Time': h['created_at'][:16]
        } for h in hist])
    )

# ── Tab 6: Notes & Memory ─────────────────────────────────────────────────────

def app_load_memory(sel):
    if not sel:
        return 'Select a candidate.', pd.DataFrame()
    cid = _cid(sel)
    mem = get_recruiter_memory(_app_state.get('job_id', ''), cid)
    if not mem:
        return 'No memory yet for this candidate.', pd.DataFrame()
    return (
        f'{len(mem)} record(s) found.',
        pd.DataFrame([{
            'Time':  m['created_at'][:16],
            'Q':     (m.get('question') or '')[:80],
            'A':     (m.get('answer') or '')[:100],
            'Notes': (m.get('recruiter_notes') or '')[:80],
        } for m in mem])
    )

def app_save_note(sel, note):
    if not sel or not note.strip():
        return '❌ Enter a note.'
    save_recruiter_memory(_app_state.get('job_id', ''), _cid(sel), '', '', note)
    return '✅ Note saved.'

# ── Tab 7: Submission ─────────────────────────────────────────────────────────

def app_gen_report(sel):
    """
    Report uses updated_score (post-assessment) as the displayed Final Score.
    All Q&A and recruiter notes are included automatically.
    """
    if not sel:
        return 'Select a candidate.', '', gr.update(interactive=False)
    cid = _cid(sel)
    jid = _app_state.get('job_id', '')
    r   = generate_report(jid, cid)

    # Inject updated_score as displayed final score
    conn = get_conn()
    row  = conn.execute(
        'SELECT updated_score FROM job_candidates WHERE job_id=? AND candidate_id=?',
        (jid, cid)
    ).fetchone()
    conn.close()
    if row:
        r['scores']['displayed_final_score'] = row['updated_score']

    _app_state['submission_report'] = r
    return '✅ Submission report ready.', format_report(r), gr.update(interactive=True)

def _format_report_with_updated_score(r):
    c  = r['candidate']
    s  = r['scores']
    # Use updated score (post-assessment) if available
    final = s.get('displayed_final_score') or s.get('updated_score') or s.get('final_score', 0)
    lines = [
        f"{'='*60}",
        "  CANDIDATE SUBMISSION REPORT",
        f"  {r['report_id']} | {r['generated_at'][:16]}",
        f"{'='*60}", "",
        f"CANDIDATE:    {c.get('name')}",
        f"Location:     {c.get('location')}  |  Exp: {c.get('experience_years')} yrs",
        f"Industry:     {c.get('industry')}",
        f"Expected CTC: ₹{c.get('expected_ctc')} LPA", "",
        "SCORES",
        f"  Match Score:      {s.get('match_score', 0):.1f}%",
        f"  Confidence Score: {s.get('confidence_score', 0):.1f}%",
        f"  Engagement Score: {s.get('engagement_score', 0):.1f}%",
        f"  Final Score:      {final:.1f}%  ← includes assessment uplift", "",
        f"SKILL MATCH ({len(r['matched_skills'])}/{len(r['matched_skills'])+len(r['missing_skills'])}):",
    ]
    for sk in r['matched_skills']:   lines.append(f'  ✓ {sk}')
    if r['missing_skills']:
        lines.append('GAPS:')
        for sk in r['missing_skills']: lines.append(f'  ✗ {sk}')
    if r.get('qa_history'):
        lines.append('\nSCREENING Q&A:')
        for qa in r['qa_history']:
            lines.append(f"  Q: {qa['q']}")
            lines.append(f"  A: {qa['a'][:120]}")
    if r.get('recruiter_notes'):
        lines.append('\nRECRUITER NOTES:')
        for n in r['recruiter_notes']:
            if n: lines.append(f'  • {n}')
    lines.append(f"\n{'='*60}")
    lines.append("Generated by AI Recruiter Copilot – TalentXO")
    return '\n'.join(lines)

def app_export_pdf():
    r = _app_state.get('submission_report')
    if not r:
        return None
    return export_pdf(r)

# ── Tab 8: Analytics ──────────────────────────────────────────────────────────

def app_record_outcome(sel, outcome, notes):
    if not sel or not outcome:
        return '❌ Select candidate and outcome.'
    record_outcome(_app_state.get('job_id', ''), _cid(sel), outcome, notes)
    return f'✅ Outcome "{outcome}" recorded.'

def app_load_analytics():
    import plotly.graph_objects as go
    data   = get_feedback_analytics()
    conn   = get_conn()
    total  = conn.execute('SELECT COUNT(*) FROM candidates').fetchone()[0]
    engaged = conn.execute(
        "SELECT COUNT(DISTINCT candidate_id) FROM job_candidates WHERE status!='Recommended'"
    ).fetchone()[0]
    conn.close()
    summary = (f"**Total Candidates in DB:** {total}\n"
               f"**Candidates Engaged:** {engaged}\n"
               f"**Reutilization Rate:** {engaged/total*100:.1f}%")
    if not data:
        fig = go.Figure(); fig.update_layout(title='No outcome data yet')
        return summary, fig
    labels = [d['outcome'] for d in data]; values = [d['count'] for d in data]
    clrs   = {'Hired':'#4caf50','Offered':'#8bc34a','Selected':'#03a9f4',
               'Interviewed':'#ff9800','Rejected':'#f44336'}
    fig    = go.Figure(go.Pie(
        labels=labels, values=values,
        marker_colors=[clrs.get(l, '#9e9e9e') for l in labels],
        hole=0.4, textinfo='label+percent+value'
    ))
    fig.update_layout(title='Hiring Outcome Distribution', height=380)
    return summary, fig

# ── Build UI ──────────────────────────────────────────────────────────────────

with gr.Blocks(theme=THEME, title='AI Recruiter Copilot') as demo:
    gr.Markdown("""
    # 🎯 AI Recruiter Copilot
    ### LangChain · Groq · FAISS Semantic Search · SQLite · Gradio
    *Reuse existing candidates before sourcing externally*
    """)

    with gr.Tabs():

        # ── Tab 1 ─────────────────────────────────────────────────────────────
        with gr.Tab('📄 1. JD Upload'):
            gr.Markdown("""
            **What this tab does:** Paste or upload a job description.
            The AI extracts Role, Skills, Experience, Location, Industry as structured JSON.
            """)
            jd_in  = gr.Textbox(label='Paste Job Description', lines=12,
                                  placeholder='Paste full JD here...')
            jd_btn = gr.Button('🔍 Parse JD with AI', variant='primary')
            jd_st  = gr.Textbox(label='Status', interactive=False)
            jd_dp  = gr.Markdown(label='Extracted Structure')
            jd_sb  = gr.Button('🔎 Search Candidates Now', variant='primary', visible=False)
            jd_btn.click(app_process_jd, jd_in, [jd_st, jd_dp, jd_sb])
            gr.Markdown('---')
            gr.Markdown('**Or record voice:**')
            voice_in  = gr.Audio(sources=['microphone'], type='filepath', label='Record Voice JD')
            voice_btn = gr.Button('🏙️ Transcribe & Parse Voice', variant='secondary')
            voice_btn.click(process_voice_jd, voice_in, [jd_st, jd_dp, jd_sb])

        # ── Tab 2 ─────────────────────────────────────────────────────────────
        with gr.Tab('🔍 2. Candidate Search'):
            gr.Markdown("""
            **What this tab does:** Runs Hybrid Search = SQL structured filter + FAISS semantic search.
            Merges both result sets, scores every candidate, returns Top 20 ranked.
            """)
            ft_in  = gr.Textbox(label='Free-Text Search (plain English)', lines=2,
                                placeholder='e.g. Senior Python developer in Mumbai with 5+ years...')
            ft_btn = gr.Button('🔎 Search by Plain Text', variant='secondary')
            ft_btn.click(run_freetext_search, ft_in, [s_st, s_tbl, s_dd])
            gr.Markdown('---')
            s_btn  = gr.Button('🔎 Run Hybrid Search (SQL + Semantic)', variant='primary')
            s_st   = gr.Textbox(label='Search Status', interactive=False)
            s_tbl  = gr.Dataframe(label='Ranked Candidates', interactive=False, wrap=True)
            s_dd   = gr.Dropdown(label='Select a Candidate to Analyze', choices=[], interactive=True)
            s_btn.click(app_run_search, outputs=[s_st, s_tbl, s_dd])
            jd_sb.click(app_run_search, outputs=[s_st, s_tbl, s_dd])

        # ── Tab 3 ─────────────────────────────────────────────────────────────
        with gr.Tab('🎯 3. Match Details (RAG)'):
            gr.Markdown("""
            **What this tab does:** Retrieves the most relevant text chunks from the
            candidate's profile (projects, experience, about) using FAISS vector search,
            then sends ONLY those chunks to the LLM to explain the match.
            No hallucination — every claim is backed by retrieved evidence.
            """)
            m_dd  = gr.Dropdown(label='Select Candidate', choices=[], interactive=True)
            m_btn = gr.Button('📊 Show Match Details', variant='primary')
            with gr.Row():
                m_mt = gr.Textbox(label='✅ Matched Requirements + Evidence', lines=10, interactive=False)
                m_ms = gr.Textbox(label='❌ Missing / Gaps', lines=10, interactive=False)
            m_sm  = gr.Textbox(label='Overall Gap Summary', interactive=False, lines=3)
            m_ev  = gr.Textbox(label='📄 Retrieved Profile Chunks (RAG Context)', lines=8, interactive=False)
            m_btn.click(app_match_details, m_dd, [m_mt, m_ms, m_sm, m_ev])
            gr.Markdown('---')
            gr.Markdown('**Ask a clarification about any gap (answered from profile by AI):**')
            clarif_in  = gr.Textbox(label='Your Question', lines=2,
                                     placeholder='e.g. Has this candidate worked on microservices?')
            clarif_btn = gr.Button('💬 Ask Clarification', variant='secondary')
            clarif_ans = gr.Markdown(label='AI Answer (from profile)')
            clarif_btn.click(ask_clarification, [m_dd, clarif_in], clarif_ans)

        # ── Tab 4 ─────────────────────────────────────────────────────────────
        with gr.Tab('📝 4. Assessment & Re-Rank'):
            gr.Markdown("""
            **What this tab does:**
            1. Generates targeted questions for gaps found in Tab 3
            2. You enter the candidate's answer
            3. AI evaluates the answer and adds a score impact (+/- points)
            4. Updated score is stored for THIS job only — master profile unchanged
            5. Same question cannot be scored twice (one-time guard)
            6. Score is always capped at 100
            """)
            a_dd  = gr.Dropdown(label='Select Candidate', choices=[], interactive=True)
            a_gb  = gr.Button('💬 Generate Screening Questions', variant='primary')
            a_qd  = gr.Markdown()
            a_qdd = gr.Dropdown(label='Select Question to Assess', choices=[], interactive=True)
            a_ans = gr.Textbox(label="Candidate's Answer", lines=5,
                                placeholder="Type or paste the candidate's response...")
            a_btn = gr.Button('🧠 Assess Answer & Update Score', variant='primary')
            a_st  = gr.Textbox(label='Status', interactive=False)
            a_rs  = gr.Markdown(label='Assessment Result')
            a_su  = gr.Markdown(label='Score Update')
            a_gb.click(app_gen_questions, a_dd, [a_qd, a_qdd])
            a_btn.click(app_assess, [a_dd, a_qdd, a_ans], [a_st, a_rs, a_su])

        # ── Tab 5 ─────────────────────────────────────────────────────────────
        with gr.Tab('📋 5. Workflow Tracker'):
            gr.Markdown("""
            **What this tab does:** Moves candidates through the hiring pipeline.
            Every status change is logged with timestamp and notes.
            Pipeline: Recommended → Contacted → Responded → Screening Complete
                      → Interview Scheduled → Selected → Offered → Hired / Rejected
            """)
            w_dd  = gr.Dropdown(label='Select Candidate', choices=[], interactive=True)
            with gr.Row():
                w_sd = gr.Dropdown(label='New Status', choices=VALID_STATUSES,
                                    value='Contacted', interactive=True)
                w_nt = gr.Textbox(label='Notes (optional)')
            w_btn = gr.Button('✅ Update Pipeline Status', variant='primary')
            w_st  = gr.Textbox(label='Update Result', interactive=False)
            w_ht  = gr.Dataframe(label='Full Status History', interactive=False)
            w_btn.click(app_update_status, [w_dd, w_sd, w_nt], [w_st, w_ht])

        # ── Tab 6 ─────────────────────────────────────────────────────────────
        with gr.Tab('🗒️ 6. Notes & Memory'):
            gr.Markdown("""
            **What this tab does:** Persistent recruiter memory per candidate per job.
            Stores all Q&A history + recruiter notes so future recruiters
            don't start from scratch on the same candidate.
            """)
            n_dd  = gr.Dropdown(label='Select Candidate', choices=[], interactive=True)
            n_lb  = gr.Button('📂 Load Memory', variant='secondary')
            n_st  = gr.Textbox(label='Status', interactive=False)
            n_mt  = gr.Dataframe(label='Memory Records (Q&A + Notes)', interactive=False)
            gr.Markdown('---')
            n_ni  = gr.Textbox(label='Add Recruiter Note', lines=3,
                                 placeholder='e.g. Strong communication, hesitant about relocation...')
            n_sb  = gr.Button('💾 Save Note')
            n_ns  = gr.Textbox(label='Save Status', interactive=False)
            n_lb.click(app_load_memory, n_dd, [n_st, n_mt])
            n_sb.click(app_save_note, [n_dd, n_ni], n_ns)

        # ── Tab 7 ─────────────────────────────────────────────────────────────
        with gr.Tab('📤 7. Submission Report'):
            gr.Markdown("""
            **What this tab does:** One-click candidate submission pack including:
            - Final Score (updated with assessment uplift)
            - Matched vs missing skills
            - Top projects
            - Full Q&A history
            - Recruiter notes
            Export as PDF for client submission.
            """)
            r_dd  = gr.Dropdown(label='Select Candidate', choices=[], interactive=True)
            r_gb  = gr.Button('📋 Generate Submission Report', variant='primary')
            r_st  = gr.Textbox(label='Status', interactive=False)
            r_tx  = gr.Textbox(label='Submission Report Preview', lines=30, interactive=False)
            r_pb  = gr.Button('📥 Export as PDF', variant='secondary', interactive=False)
            r_fi  = gr.File(label='Download PDF')
            r_gb.click(app_gen_report, r_dd, [r_st, r_tx, r_pb])
            r_pb.click(app_export_pdf, outputs=r_fi)

        # ── Tab 8 ─────────────────────────────────────────────────────────────
        with gr.Tab('📊 8. Analytics & Feedback'):
            gr.Markdown("""
            **What this tab does:** Closes the feedback loop.
            Record outcomes (Hired/Rejected etc.) to measure:
            - Candidate reutilization rate
            - Recommendation accuracy
            - Pipeline conversion rates
            """)
            with gr.Row():
                fb_dd = gr.Dropdown(label='Select Candidate', choices=[], interactive=True)
                fb_oc = gr.Dropdown(label='Outcome',
                                     choices=['Rejected','Interviewed','Selected','Offered','Hired'],
                                     interactive=True)
                fb_nt = gr.Textbox(label='Notes')
            fb_rb  = gr.Button('📝 Record Outcome', variant='primary')
            fb_st  = gr.Textbox(label='Status', interactive=False)
            gr.Markdown('---')
            an_btn = gr.Button('📊 Load Analytics Dashboard')
            an_mt  = gr.Markdown()
            an_ch  = gr.Plot(label='Outcome Distribution')
            fb_rb.click(app_record_outcome, [fb_dd, fb_oc, fb_nt], fb_st)
            an_btn.click(app_load_analytics, outputs=[an_mt, an_ch])

    # ── Sync all candidate dropdowns after search ─────────────────────────────
    s_btn.click(app_run_search, outputs=[s_st, s_tbl, s_dd]).then(
        fn=lambda: [gr.update(choices=_candidate_choices())] * 7,
        outputs=[m_dd, a_dd, w_dd, n_dd, r_dd, fb_dd, s_dd]
    )

demo.launch(share=True, debug=False)